# Efficient ResNet on CIFAR-10 — Colab runner

1. 把数据压缩包 `kaggle_data.zip`（`train_images/`、`test_images/`、`train_labels.csv`、`test.csv`、`sample_submission.csv`）上传到 Google Drive 的 `MyDrive/` 根目录。
2. 菜单 **Runtime → Change runtime type → GPU**（T4 即可）。
3. 每次实验改一下 `RUN_NAME`（和训练参数），从上到下运行。
4. 每次实验的结果都保存在 Drive 的 `MyDrive/kaggle_DL_output/runs/<RUN_NAME>/`，互不覆盖，断线也不会丢：
   `config.json`、`train_log.txt`、`history.json`、`summary.json`、`best.pt`、`submission.csv`

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, 'CUDA:', torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL   = 'https://github.com/yzeng018-cpu/DeepLearningProject1.git'
DATA_ZIP   = '/content/drive/MyDrive/kaggle_data.zip'           # 上传数据 zip 的位置
RUNS_DIR   = '/content/drive/MyDrive/kaggle_DL_output/runs'     # 所有实验结果的保存位置
WORK_DIR   = '/content/project'                                 # 本地磁盘，读图片比 Drive 快得多

RUN_NAME   = 'run02_baseline_e100'                              # ← 每次实验改一个新名字

In [ ]:
import os
os.makedirs(RUNS_DIR, exist_ok=True)
!rm -rf {WORK_DIR} && git clone -q {REPO_URL} {WORK_DIR}
!unzip -q "{DATA_ZIP}" -d {WORK_DIR}
%cd {WORK_DIR}
!ls
!ls train_images | wc -l && ls test_images | wc -l

In [ ]:
# 检查参数量 (<= 5M)
!python model.py

In [ ]:
# 训练：90% train / 10% val，按验证准确率保存 best.pt
# 改超参数直接加参数，如 --epochs 200 --weight-decay 1e-3；用全部 5 万张训练时加 --val-frac 0
!python train.py --epochs 100 --run-name {RUN_NAME} --runs-dir "{RUNS_DIR}"

In [ ]:
# 生成提交文件 -> RUNS_DIR/RUN_NAME/submission.csv（test_images 只用于推理）
!python predict.py --run-name {RUN_NAME} --runs-dir "{RUNS_DIR}"
!head -5 "{RUNS_DIR}/{RUN_NAME}/submission.csv"

In [ ]:
# 可选：下载这次的 submission.csv 到本地
from google.colab import files
files.download(f'{RUNS_DIR}/{RUN_NAME}/submission.csv')

In [ ]:
# 对比所有实验（Markdown 表格，可直接贴进报告）
!python compare_runs.py --runs-dir "{RUNS_DIR}"